# Lektion 17: Gedächtnis, Checkpointer & Freigabe



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Fortgeschritten |
| **Voraussetzungen** | Lektion 15–16 |
| **Du baust** | Einen Agenten mit thread-Gedächtnis und einen isolierten Freigabe-Workflow mit Beweisen |

## 🎯 Nach dieser Lektion kannst du:

- Kurzzeit-Gedächtnis über mehrere Aufrufe mit **Checkpointer + `thread_id`**

- nachweisen, dass eine andere `thread_id` ein leeres Gedächtnis bedeutet

- die `interrupt`/`Command(resume=...)`-Mechanik isoliert verstehen

- erklären, warum `interrupt` ohne Checkpointer nicht funktioniert

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Der Vorgangsordner

**WARUM:** Ein Agent, der bei jedem Aufruf vergisst, nervt: "Wie war nochmal Ihr Name?"

**WAS:** Der **Checkpointer** speichert den Zustand **pro Gespräch** (`thread_id`). **Metapher:** der Vorgangsordner im Büro – neues Personal am Schalter kann weitermachen, weil alles drinsteht. Ein **anderer** Ordner (`thread_id`) ist ein anderes Gespräch: frisch und leer.

**Und die Freigabe?** `interrupt()` pausiert den Lauf **im Backend**; `Command(resume=True/False)` setzt ihn mit deiner Entscheidung fort. Das kennst du schon aus Lektion 15 – hier siehst du die Mechanik noch einmal isoliert, als handgebauter Agent (damit du weißt, was `create_agent` intern tut).

In [2]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt
from typing import Annotated, TypedDict

from lernakademie.llm_client import get_chat_model

NOTIZEN = []   # simulierter Notizspeicher

@tool
def notiz_anlegen(text: str) -> str:
    """Legt eine normale Notiz an."""
    NOTIZEN.append(text)
    return f"Notiz angelegt ({len(text)} Zeichen)."

@tool
def geheime_notiz(text: str) -> str:
    """Legt eine geheime Notiz an - erfordert vorher die Freigabe eines Menschen."""
    freigabe = interrupt({          # <- Lauf PAUSIERT hier (vor der Aktion!)
        "frage": "Soll diese geheime Notiz wirklich angelegt werden?",
        "text": text,
    })
    if freigabe:
        NOTIZEN.append(f"[GEHEIM] {text}")
        return "Geheime Notiz angelegt - Freigabe erteilt."
    return "Aktion abgebrochen - keine Freigabe."

class AgentZustand(TypedDict):
    messages: Annotated[list, add_messages]   # Reduzierer: Liste WAECHST pro Zug

modell = get_chat_model(temperature=0.0).bind_tools([notiz_anlegen, geheime_notiz])
tools = ToolNode([notiz_anlegen, geheime_notiz])

def model_node(state):
    return {"messages": [modell.invoke(state["messages"])]}

def weiche(state):
    letzte = state["messages"][-1]
    return "tools" if getattr(letzte, "tool_calls", None) else END

g = StateGraph(AgentZustand)
g.add_node("model", model_node)
g.add_node("tools", tools)
g.add_edge(START, "model")
g.add_conditional_edges("model", weiche, {"tools": "tools", END: END})
g.add_edge("tools", "model")            # zurueck zum Modell -> LOOP
agent = g.compile(checkpointer=MemorySaver())   # ohne Checkpointer kein interrupt!
print("Agent bereit (mit Checkpointer).")

Agent bereit (mit Checkpointer).


## ✏️ Deine Aufgabe: Gedächtnis-Beweis und Freigabe-Roundtrip

Du führst den Agenten in zwei Threads und durch einen kompletten Freigabe-Zyklus – mit Beweisen.

**Anforderungen:**

- **Gedächtnis:** Thread `"gespraech-1"`: erzähle deinen Namen, frage dann "Wie heiße ich?" – zähle die Nachrichten im Thread-State (`agent.get_state(config)`).

- **Kontrolle:** dieselbe Frage in Thread `"anderes-gespraech"` – das Gedächtnis ist leer.

- **Freigabe:** Bitte im Thread `"freigabe"` um eine geheime Notiz. Der Lauf pausiert am Interrupt – lies die Daten aus `__interrupt__`, setze mit `Command(resume=True)` fort.

- **Beweise:** `NOTIZEN` enthält danach einen `[GEHEIM]`-Eintrag.

**Schreibe deinen Code in die folgende Zelle:**

In [3]:
config1 = {"configurable": {"thread_id": "gespraech-1"}}
agent.invoke({"messages": [HumanMessage(content="Hallo! Ich heisse Alex.")]}, config1)
agent.invoke({"messages": [HumanMessage(content="Wie heisse ich?")]}, config1)
zustand1 = agent.get_state(config1)
print("Nachrichten im Thread 1:", len(zustand1.values["messages"]))

config2 = {"configurable": {"thread_id": "anderes-gespraech"}}
agent.invoke({"messages": [HumanMessage(content="Wie heisse ich?")]}, config2)
zustand2 = agent.get_state(config2)
print("Nachrichten im Thread 2 (frisch):", len(zustand2.values["messages"]))

config_f = {"configurable": {"thread_id": "freigabe"}}
lauf = agent.invoke({"messages": [HumanMessage(
    content="Lege bitte die geheime Notiz 'Gehaltsliste' an.")]}, config_f)
if lauf.get("__interrupt__"):
    daten = lauf["__interrupt__"][0].value
    print("INTERRUPT:", daten["frage"], "|", daten["text"])
    fortsetzung = agent.invoke(Command(resume=True), config_f)
    print("Nach Freigabe:", fortsetzung["messages"][-1].content[:80])
print("Notizen:", NOTIZEN)

Nachrichten im Thread 1: 4
Nachrichten im Thread 2 (frisch): 2
INTERRUPT: Soll diese geheime Notiz wirklich angelegt werden? | Lege bitte die geheime Notiz 'Gehaltsliste' an.
Nach Freigabe: [OFFLINE] Finale Antwort auf Basis der Tool-Ergebnisse: Geheime Notiz angelegt -
Notizen: ["[GEHEIM] Lege bitte die geheime Notiz 'Gehaltsliste' an."]


**💡 Warum diese Loesung funktioniert:**

Der Checkpointer hält den Zustand pro `thread_id` – deshalb funktioniert das Gedächtnis UND die Pause: Der angehaltene Lauf wird gesichert und mit `resume` exakt fortgesetzt. **Ohne Checkpointer wäre beides unmöglich.** Genau diese Mechanik läuft im Finale über HTTP: der Browser ist nur die Fernbedienung, pausiert wird im Backend.

### ✅ Richtig, wenn:

- [ ] Im Thread 1 wächst der State über beide Aufrufe (Nachrichtenzahl steigt).

- [ ] Thread 2 startet mit leerem Gedächtnis.

- [ ] Der geheime Eintrag existiert erst NACH `resume=True`.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [4]:
zustand1 = agent.get_state(config1)
assert len(zustand1.values["messages"]) >= 3, "Thread 1 sollte den Verlauf speichern."
assert any("[GEHEIM]" in n for n in NOTIZEN), "resume=True muss die Notiz anlegen."
print("Alle Prüfungen bestanden! ✅")

Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Zwei Aufrufe im selben Thread: einfach zweimal `agent.invoke({"messages": [HumanMessage(content=...)]}, config1)`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Interrupt-Daten: `lauf["__interrupt__"][0].value` (Frage + Text).
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Fortsetzen: `agent.invoke(Command(resume=True), config_f)` – **identisches** config!
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Was wäre "Langzeit-Gedächtnis über Monate" – und warum ist ein thread-Checkpointer es noch nicht?

2. Welche Aktionen würdest du NIE ohne Freigabe automatisieren – und welche gezielt doch?

## ➡️ Weiter

**Weiter mit Lektion 18:** `notebooks/09_agents/18_supervisor_und_parallelitaet.ipynb` – wenn ein Agent nicht reicht: Parallelität, Subgraphs und der Supervisor.